In [1]:
import numpy as np
from numpy.linalg import norm
import math, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [29]:
#spatial funcs for WoS

def closestPoint(v, s):
    u = s[1] - s[0]
    rate = max(0, min(u.dot(v - s[0])/u.dot(u), 1))
    return (s[0] + rate*(s[1] - s[0]))

def shortestDistance(v, segments):
    r = float("inf")
    for s in segments:
        u = closestPoint(v, s)
        if norm(v - u) < r:
            r = norm(v - u)
    return r

In [419]:
#WoS solver
'''record info for all the valid paths
    a list of list having path-steps (start -> boundary end) (dynamic length), 
    a list of list having radius 
    a list of boundary results, 
    a final local estimation
'''
def solver(v, segments, g, eps = 0.01, nWalks = 20000, maxSteps = 16, maxR = 0.2):
    vWalks = 0   
    sumEst = 0
    paths = []
    boundary = []
    for i in range(0, nWalks):
        x0 = v
        path = []
        for step in range(0, maxSteps): 
            path.append(x0)
            r = min([shortestDistance(x0, segments), maxR])
            if r < eps: 
                sumEst += g(x0)
                boundary.append(g(x0))
                vWalks += 1
                paths.append(path)
                #print("walk: " + str(vWalks) + " hit " + str(x0) + " boundary " + str(g(x0)))
                break
            theta = random.uniform(0, 2*math.pi)
            x0 = x0 + np.array([r * math.cos(theta), r * math.sin(theta)])
        
    if vWalks == 0:
        return 0
    return paths, boundary, sumEst/vWalks

In [420]:
# set up the problem
segments = [np.array([[-1, -1], [-1, 1]]), np.array([[-1, -1], [1, -1]]), np.array([[1, -1], [1, 1]]), np.array([[-1, 1], [1, 1]])]
v = np.array([0, 0])
def boundary(v):
    return v[1] * v[0]

In [421]:
paths, boundary, result = solver(v, segments, boundary)
print(len(paths))

3433


In [422]:
class ZNN(keras.Model):
    def __init__(self, units):
        super(ZNN, self).__init__()
        self.layer1 = keras.layers.Dense(units[0])
        self.layer2 = keras.layers.Dense(units[1])
        self.layer3 = keras.layers.Dense(units[2])
        #self.bn1 = keras.layers.BatchNormalization()
        #self.bn2 = keras.layers.BatchNormalization()
        self.outputs = layers.Dense(2)

    def call(self, input_tensor, training):
        x = self.layer1(input_tensor)
        #x = self.bn1(x, training)
        x = tf.nn.relu(x)
        x = self.layer2(x)
        x = tf.nn.relu(x)
        x = self.layer3(x)
        x = tf.nn.relu(x)
        x = self.outputs(x)
        #print(x)
        return x

In [429]:
class NNsolver(keras.Model):
    def __init__(self, est, paths, boundary):
        super(NNsolver, self).__init__()
        #self.paths, self.radius, self.boundary, self.est = data
        #self.est = est
        self.est = 0
        self.paths = paths
        self.boundary = boundary
        self.model = ZNN([32, 64, 128])
        #self.model = ZNN([64, 128])
        self.optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=1e-4 , epsilon=1e-8)

    def loss_fn(self, y_pred, y):
        delta = y_pred - y
        loss = tf.square(delta)
        return loss

    def train_step(self, x, training):
        y_pred = self.est
        for i in range(len(x) - 1):
            z = self.model(x[i].reshape(1,2).astype("float32"), training) #obtain gradients for steps
            y_pred += tf.reduce_sum(z * np.array(x[i + 1] - x[i]), 1, keepdims=True) #accumulating
        return y_pred

    def train(self): 
        training_history = []
        for i in range(len(self.paths)):
            with tf.GradientTape() as tape:
                y_pred = self.train_step(np.array(self.paths[i]), training = True)
                loss = self.loss_fn(y_pred,self.boundary[i])

            training_vars = self.model.trainable_variables
            #print(loss)
            grad = tape.gradient(loss, training_vars)
            self.optimizer.apply_gradients(zip(grad, training_vars))
            
            training_history.append([i, loss])

        return training_history

    def test1(self, v):
        return self.model(v.reshape(1,2).astype("float32"), training = True)

    def test2(self, path):
        return self.train_step(path, training = True)

    def accuracy1(self, i):
        return abs((self.test2(self.paths[i]) - self.boundary[i])/self.boundary[i])

    def accuracy2(self):
        sum = 0
        for i in range(100):
            sum += self.accuracy1(i)
        return sum/100


In [435]:
training = NNsolver(result, paths, boundary)

training_history = training.train() #no epoch

In [436]:
print(training.test1(np.array([-0.5, -1])))

tf.Tensor([[-0.9695223  -0.48832408]], shape=(1, 2), dtype=float32)


In [437]:
print(training.test2(paths[4]))
print(paths[4])
print(boundary[4])

tf.Tensor([[-0.08804712]], shape=(1, 1), dtype=float32)
ListWrapper([array([0, 0]), array([-0.05394674, -0.19258699]), array([-0.15735334, -0.02139389]), array([-0.31021344,  0.10757815]), array([-0.50831371,  0.08007754]), array([-0.53831949, -0.11765878]), array([-0.7380669 , -0.10761038]), array([-0.80690677,  0.08016894]), array([-0.98758372,  0.1482926 ]), array([-0.9806068 ,  0.13802194]), array([-0.9987687 ,  0.13122182])])
-0.13106025075560426


In [438]:
print(training.accuracy1(4))

tf.Tensor([[0.32819363]], shape=(1, 1), dtype=float32)


In [439]:
print(training.accuracy2())

tf.Tensor([[0.4083708]], shape=(1, 1), dtype=float32)
